# Notebook 11: 简化 Video UNet（时空建模演示）

**目标**：实现一个 toy video diffusion 模型，观察 spatial + temporal attention 因子分解的工作。

**前置**：L14

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. 构造 toy video 数据

用 moons 数据集 "旋转动画"：8 帧，每帧旋转 45°，模拟一段视频。

In [ ]:
def make_rotating_moons(n_videos=500, n_frames=8):
    """每个 video 是 moons 数据集做 n_frames 步连续旋转"""
    videos = []
    for _ in range(n_videos):
        x, _ = make_moons(64, noise=0.05)  # 64 个点
        x = torch.tensor(x, dtype=torch.float32)
        frames = [x]
        for f in range(1, n_frames):
            theta = np.pi / 4 * f  # 旋转 45° each frame
            R = torch.tensor([[np.cos(theta), -np.sin(theta)],
                             [np.sin(theta), np.cos(theta)]], dtype=torch.float32)
            frames.append(x @ R.T)
        videos.append(torch.stack(frames))  # (T, N_points, 2)
    return torch.stack(videos)

videos = make_rotating_moons(500, 8).to(device)
print(f'videos shape: {videos.shape}')  # (500, 8, 64, 2)

# Show one video
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for f in range(8):
    axes[f].scatter(videos[0, f, :, 0].cpu(), videos[0, f, :, 1].cpu(), s=10)
    axes[f].set_aspect('equal'); axes[f].set_xlim(-2, 2); axes[f].set_ylim(-2, 2)
    axes[f].set_title(f'frame {f}')
plt.tight_layout(); plt.show()

## 2. Video Network: Spatial + Temporal Attention

In [ ]:
class VideoTransformer(nn.Module):
    """对 (B, T, N, 2) 数据：spatial attn within frame, temporal attn within point"""
    def __init__(self, dim=64, n_layers=3):
        super().__init__()
        self.proj_in = nn.Linear(2, dim)
        self.t_emb = nn.Sequential(nn.Linear(1, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.f_emb = nn.Embedding(16, dim)  # frame index up to 16
        self.spatial_attn = nn.ModuleList([
            nn.TransformerEncoderLayer(dim, 4, dim_feedforward=128, batch_first=True)
            for _ in range(n_layers)])
        self.temporal_attn = nn.ModuleList([
            nn.TransformerEncoderLayer(dim, 4, dim_feedforward=128, batch_first=True)
            for _ in range(n_layers)])
        self.proj_out = nn.Linear(dim, 2)
    def forward(self, x, t):
        # x: (B, T, N, 2)
        B, T, N, _ = x.shape
        h = self.proj_in(x)  # (B, T, N, D)
        # 加 timestep + frame embedding
        t_e = self.t_emb(t.float().unsqueeze(-1) / 1000.0).view(B, 1, 1, -1)
        f_e = self.f_emb(torch.arange(T, device=x.device)).view(1, T, 1, -1)
        h = h + t_e + f_e
        for s_attn, t_attn in zip(self.spatial_attn, self.temporal_attn):
            # Spatial：每 frame 独立 attention over N points
            h_sp = h.reshape(B * T, N, -1)
            h_sp = s_attn(h_sp)
            h = h_sp.reshape(B, T, N, -1)
            # Temporal：每 point 独立 attention over T frames
            h_tp = h.permute(0, 2, 1, 3).reshape(B * N, T, -1)
            h_tp = t_attn(h_tp)
            h = h_tp.reshape(B, N, T, -1).permute(0, 2, 1, 3)
        return self.proj_out(h)  # (B, T, N, 2)

model = VideoTransformer().to(device)
print(f'params: {sum(p.numel() for p in model.parameters())/1e3:.1f}K')

## 3. 训练（DDPM on videos）

In [ ]:
T_diff = 1000
betas = torch.linspace(1e-4, 0.02, T_diff).to(device)
alphas = 1 - betas
ac = alphas.cumprod(0)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
for step in range(2000):
    idx = torch.randint(0, len(videos), (32,))
    x = videos[idx]  # (32, 8, 64, 2)
    t = torch.randint(0, T_diff, (x.shape[0],), device=device)
    eps = torch.randn_like(x)
    xt = ac[t].view(-1,1,1,1).sqrt() * x + (1-ac[t]).view(-1,1,1,1).sqrt() * eps
    loss = F.mse_loss(model(xt, t), eps)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 400 == 0:
        print(f'step {step}: loss={np.mean(losses[-100:]):.4f}')

## 4. 采样

In [ ]:
@torch.no_grad()
def sample_video(n_steps=50):
    model.eval()
    x = torch.randn(1, 8, 64, 2, device=device)
    step = T_diff // n_steps
    for ti in reversed(range(0, T_diff, step)):
        t = torch.full((1,), ti, device=device, dtype=torch.long)
        eps = model(x, t)
        x0 = (x - (1-ac[ti]).sqrt() * eps) / ac[ti].sqrt()
        x0 = x0.clamp(-3, 3)
        if ti - step >= 0:
            x = ac[ti-step].sqrt() * x0 + (1 - ac[ti-step]).sqrt() * eps
        else:
            x = x0
    return x[0]

torch.manual_seed(7)
video_gen = sample_video(50).cpu()

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for f in range(8):
    axes[f].scatter(video_gen[f, :, 0], video_gen[f, :, 1], s=10, c='blue')
    axes[f].set_aspect('equal'); axes[f].set_xlim(-2, 2); axes[f].set_ylim(-2, 2)
    axes[f].set_title(f'gen frame {f}')
plt.suptitle('Generated rotating moons')
plt.tight_layout(); plt.show()

## 5. 验证时间一致性

对于真实 video：相邻帧应有 45° 旋转关系。生成 video 是否保持？

In [ ]:
# 计算相邻帧之间的"旋转匹配度"
def frame_to_frame_distance(video):
    """在 video 中，假设 frame_{i+1} 是 frame_i 旋转 45°"""
    theta = np.pi / 4
    R = torch.tensor([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta), np.cos(theta)]])
    diffs = []
    for i in range(len(video) - 1):
        expected = video[i] @ R.T
        diff = (video[i+1] - expected).norm(dim=-1).mean()
        diffs.append(diff.item())
    return diffs

print('Generated:')
for d in frame_to_frame_distance(video_gen):
    print(f'  {d:.3f}')
print('\nReal:')
for d in frame_to_frame_distance(videos[0].cpu()):
    print(f'  {d:.3f}')

## 思考题

1. 把 temporal_attn 全部移除（只剩 spatial），训练，观察 video 一致性退化
2. 加入 `frame index` 之外的 `relative time` embedding (RoPE-style)，观察泛化到不同长度 video
3. 实现 image-to-video：第一帧给定（mask 掉），后续帧生成
4. 把数据集换成 "line drawing → animation"（更复杂的运动）